In [6]:
import pandas as pd
import numpy as np
# Load the input data
input_df = pd.read_excel('DATest_2.xlsx', sheet_name='Age_Input')
input_df

,User Id,Age,Keyword
0,00831062-d45b-11ee-8482-128e718ba88f,25-34,Total
1,00ac49e8-d45a-11ee-8482-128e718ba88f,35-45,Total
2,00fd1031-d466-11ee-8482-128e718ba88f,Above 50,Total
3,011c0f25-d464-11ee-8482-128e718ba88f,Above 50,Total
4,01bc1864-d460-11ee-8482-128e718ba88f,Below 25,Total
...,...,...,...
1933,fbd0ca7d-d466-11ee-8482-128e718ba88f,25-34,Rejector
1934,fc2bce87-d467-11ee-8482-128e718ba88f,Below 25,Rejector
1935,fc3b09c3-d465-11ee-8482-128e718ba88f,Below 25,Rejector
1936,fd339b8a-d469-11ee-8482-128e718ba88f,35-45,Rejector


In [8]:
# Create a pivot table to count occurrences of each Keyword by Age
pivot_counts = pd.pivot_table(input_df, 
                             index='Age', 
                             columns='Keyword', 
                             values='User Id', 
                             aggfunc='count', 
                             fill_value=0)

# Reorder the age groups to match desired output
age_order = ['Below 25', '25-34', '35-45', '46-50', 'Above 50']
pivot_counts = pivot_counts.reindex(age_order)

# Add missing age group '46-50' with zeros if it doesn't exist
if '46-50' not in pivot_counts.index:
    pivot_counts.loc['46-50'] = 0

# Calculate grand totals for each keyword and age group
pivot_counts['Grand Total'] = pivot_counts.sum(axis=1)

# Reorder columns to match desired output
pivot_counts = pivot_counts[['Intender', 'Owner', 'Rejector', 'Grand Total']]

# Calculate the total counts for percentage calculations
total_intender = pivot_counts['Intender'].sum()
total_owner = pivot_counts['Owner'].sum()
total_rejector = pivot_counts['Rejector'].sum()
total_grand = pivot_counts['Grand Total'].sum()

# Calculate percentages
percentages = pivot_counts.copy()
percentages['Intender'] = (percentages['Intender'] / total_intender * 100).round(0).astype(int).astype(str) + '%'
percentages['Owner'] = (percentages['Owner'] / total_owner * 100).round(0).astype(int).astype(str) + '%'
percentages['Rejector'] = (percentages['Rejector'] / total_rejector * 100).round(0).astype(int).astype(str) + '%'
percentages['Grand Total'] = (percentages['Grand Total'] / total_grand * 100).round(0).astype(int).astype(str) + '%'

# Reset index to make Age a column
pivot_counts = pivot_counts.reset_index()
percentages = percentages.reset_index()

# Merge counts and percentages
result = pivot_counts.merge(percentages, on='Age', suffixes=('', '.1'))

# Add the totals row
totals_row = pd.DataFrame({
    'Age': [''],
    'Intender': [pivot_counts['Intender'].sum()],
    'Owner': [pivot_counts['Owner'].sum()],
    'Rejector': [pivot_counts['Rejector'].sum()],
    'Grand Total': [pivot_counts['Grand Total'].sum()],
    'Intender.1': [pivot_counts['Intender'].sum()],
    'Owner.1': [pivot_counts['Owner'].sum()],
    'Rejector.1': [pivot_counts['Rejector'].sum()],
    'Grand Total.1': [pivot_counts['Grand Total'].sum()]
})

result = pd.concat([result, totals_row], ignore_index=True)

# Reorder columns to match desired output
column_order = ['Age', 'Intender', 'Owner', 'Rejector', 'Grand Total', 
                'Intender.1', 'Owner.1', 'Rejector.1', 'Grand Total.1']
result = result[column_order]

# Rename columns to remove the .1 suffix for display (but keep in CSV)
result_display = result.copy()
result_display.columns = ['Age', 'Intender', 'Owner', 'Rejector', 'Grand Total', 
                         'Intender', 'Owner', 'Rejector', 'Grand Total']

# %% [markdown]
# ## Generate Output File

# %%
# Save to CSV with the exact format (keeping original column names with .1)
result.to_csv('Age_DesiredOutput.csv', index=False)

# Display the result to verify (with clean column names)
result_display

,Age,Intender,Owner,Rejector,Grand Total,Intender,Owner,Rejector,Grand Total
0,Below 25,33,62,36,262,12%,14%,17%,14%
1,25-34,142,166,105,826,50%,39%,49%,45%
2,35-45,80,123,67,540,28%,29%,31%,29%
3,46-50,11,24,2,74,4%,6%,1%,4%
4,Above 50,17,54,6,154,6%,13%,3%,8%
5,,283,429,216,1856,283,429,216,1856
